In [ ]:
# ================================================================================
# PhyloCNN testing and conformal calibration under the BDEIVH model
# ================================================================================
#
# This notebook loads a previously trained PhyloCNN model, generates predictions
# for the calibration and test datasets, applies conformalized quantile
# regression (CQR), and exports the test results.
# ================================================================================

import os
import pandas as pd
import numpy as np
import tensorflow as tf
import keras

from keras.saving import register_keras_serializable
from keras.models import model_from_json

from tensorflow.keras.models import model_from_json
from keras.src.saving import serialization_lib

QUANTILES = (
    0.025,  # lower
    0.5,    # median
    0.975   # upper
)


In [2]:
# Custom pinball loss function for simultaneous quantile regression.
# The model predicts three quantiles (2.5%, 50%, and 97.5%) for each target
# parameter. This loss penalizes under- and over-estimation asymmetrically,
# according to the specified quantiles.

# Input shapes:
#
# y_true:
#     (batch_size, n_targets)
#
# y_pred (raw network output):
#     (batch_size, n_targets × n_quantiles)
#
# Example for six target parameters:
#
# y_true.shape = (batch_size, 6)
# y_pred.shape = (batch_size, 18)
#
# After reshaping:
#
# y_pred.shape = (batch_size, 6, 3)
#
# where the last dimension corresponds to:
# [lower, median, upper]

@register_keras_serializable(package="bdeissct_dl", name="pinball_loss")
def pinball_loss(y_true, y_pred):
    # Convert the tuple of quantiles into a TensorFlow constant.
    quantiles_tensor = tf.constant(QUANTILES, dtype=tf.float32)

    # Number of target parameters predicted by the model.
    n_targets = tf.shape(y_true)[-1]

    # Number of quantiles predicted for each target.
    n_quantiles = len(QUANTILES)

    # Reshape predictions from (batch, n_targets × n_quantiles)
    # to (batch, n_targets, n_quantiles).
    y_pred = tf.reshape(y_pred, (-1, n_targets, n_quantiles))

    # Add a singleton dimension so that each true value can be compared
    # simultaneously with its three predicted quantiles through
    # TensorFlow broadcasting.
    y_true = tf.expand_dims(y_true, axis=-1)

    # Compute prediction errors.
    #
    # Positive error:
    #     prediction < true value
    #
    # Negative error:
    #     prediction > true value
    errors = y_true - y_pred

    # Compute the pinball loss for each quantile.

    # Pinball (quantile) loss:
    #
    # rho_q(e) = max(qe, (q-1)e)
    #
    # where:
    #
    #     e = y_true - y_pred
    #     q = quantile level
    #
    # The asymmetric penalty encourages each output neuron to estimate
    # the specified conditional quantile.
    loss_per_quantile = tf.maximum(
        quantiles_tensor * errors,
        (quantiles_tensor - 1.0) * errors
    )

    # Return the average loss across all targets, quantiles, and samples.
    return tf.reduce_mean(loss_per_quantile)

NameError: name 'register_keras_serializable' is not defined

In [ ]:
# Transform unconstrained network outputs into positive,
# ordered, and non-crossing quantile predictions.
#
# For each target parameter, the network initially produces:
#
#     raw_lower
#     raw_gap_to_median
#     raw_gap_to_upper
#
# The final outputs are constructed as:
#
#     lower  = softplus(raw_lower)
#     median = lower + softplus(raw_gap_to_median)
#     upper  = median + softplus(raw_gap_to_upper)
#
# Therefore:
#
#     0 <= lower <= median <= upper
#
# The final outputs continue to represent the predicted
# 2.5%, 50%, and 97.5% quantiles.

@register_keras_serializable(
    package="bdeissct_dl",
    name="ordered_positive_quantiles"
)
def ordered_positive_quantiles(raw, n_targets=6):

    # Reshape:
    #
    # (batch_size, 18)
    # ->
    # (batch_size, 6, 3)
    raw = tf.reshape(
        raw,
        (-1, n_targets, len(QUANTILES))
    )

    # Extract the three unrestricted intermediate values.
    raw_lower = raw[..., 0]
    raw_gap_to_median = raw[..., 1]
    raw_gap_to_upper = raw[..., 2]

    # Transform the first unrestricted output into a non-negative
    # lower quantile using the Softplus function.
    lower = tf.nn.softplus(raw_lower)

    # Transform the second unrestricted output into a non-negative
    # increment from the lower quantile to the median.
    median = lower + tf.nn.softplus(
        raw_gap_to_median
    )

    # Non-negative increment from median to upper.
    upper = median + tf.nn.softplus(
        raw_gap_to_upper
    )

    # Arrange the final predictions in the order expected
    # by the pinball loss: [lower, median, upper].
    quantiles = tf.stack(
        [lower, median, upper],
        axis=-1
    )

    # Return:
    #
    # (batch_size, 6, 3)
    # ->
    # (batch_size, 18)
    return tf.reshape(
        quantiles,
        (-1, n_targets * len(QUANTILES))
    )

In [ ]:
# Load the test and calibration parameters and tree encodings.

param_test = pd.read_csv(
    './datasets/250_350_test_merged_all_OROV.log', sep=','
)

param_calibration = pd.read_csv(
    './datasets/250_350_calibrate_merged_all_OROV.log', sep=','
)

encoding_test = pd.read_csv(
    './datasets/250_350_test_merged_all_OROV.nwk_encoded',
    sep="\t",
    header=0,
    index_col=0
).values.astype('float32').reshape(-1, 800, 11)

encoding_calibration = pd.read_csv(
    './datasets/250_350_calibrate_merged_all_OROV.nwk_encoded',
    sep="\t",
    header=0,
    index_col=0
).values.astype('float32').reshape(-1, 800, 11)

# Match parameter rows to encoding rows.
param_test = param_test.iloc[:encoding_test.shape[0]].reset_index(drop=True)
param_calibration = param_calibration.iloc[
    :encoding_calibration.shape[0]
].reset_index(drop=True)

print(f"Test: {encoding_test.shape[0]} trees | {param_test.shape[0]} parameter rows")
print(
    f"Calibration: {encoding_calibration.shape[0]} trees | "
    f"{param_calibration.shape[0]} parameter rows"
)

In [ ]:
# Define targets and prepare the test and calibration labels.

target_1 = "Rvh"        # Transmission from vectors to humans
target_2 = "Rhv"        # Transmission from humans to vectors
target_3 = "Rhh"        # Transmission among humans
target_4 = "d_v"        # Total duration of the infection in vectors
target_5 = "d_h"        # Total duration of the infection in humans
target_6 = "v2h_ratio"  # Ratio of vectors and humans

target_cols = [target_1, target_2, target_3, target_4, target_5, target_6]

# Extract the tree-specific rescaling factors.
param_test['resc_factor'] = encoding_test[:, -1, -1]
param_calibration['resc_factor'] = encoding_calibration[:, -1, -1]

# Rescale time-dependent targets.
for key in (target_4, target_5):
    param_test[key] = param_test[key] / param_test['resc_factor']
    param_calibration[key] = (
        param_calibration[key] / param_calibration['resc_factor']
    )

# Remove the final row containing the rescaling factor.
encoding_test = np.delete(encoding_test, -1, axis=1)
encoding_calibration = np.delete(encoding_calibration, -1, axis=1)

Y_test = pd.DataFrame(param_test[target_cols]).reset_index(drop=True)
Y_calibration = pd.DataFrame(
    param_calibration[target_cols]
).reset_index(drop=True)

print("Y test:", Y_test.shape)
print("Y calibration:", Y_calibration.shape)


In [ ]:
# Add the tree-level sampling probability (p) as an additional feature.

n_rows_test = encoding_test.shape[1]
n_rows_calibration = encoding_calibration.shape[1]

samp_proba_list_test = np.asarray(param_test['p'])
encoding_test = np.concatenate(
    (
        encoding_test,
        np.repeat(samp_proba_list_test, n_rows_test).reshape(
            -1, n_rows_test, 1
        )
    ),
    axis=2
)

samp_proba_list_calibration = np.asarray(param_calibration['p'])
encoding_calibration = np.concatenate(
    (
        encoding_calibration,
        np.repeat(
            samp_proba_list_calibration, n_rows_calibration
        ).reshape(-1, n_rows_calibration, 1)
    ),
    axis=2
)

print("Test encoding:", encoding_test.shape)
print("Calibration encoding:", encoding_calibration.shape)


In [ ]:
# This function converts each encoded tree into two fixed-length channels:
# one for leaves and one for internal nodes.
#
# Leaves and internal nodes are sorted by distance from the root and
# zero-padded to a maximum of 400 rows per channel.

def encode_pad_0s_rootage_single(enc, max_len=400):
    enc_pad = []

    for i in range(enc.shape[0]):
        # Leaves: column 3 == 1.
        leaves = enc[i][enc[i, :, 3] == 1]
        leaves = leaves[np.argsort(leaves[:, 1])]
        leaves = np.pad(leaves, [(0, (max_len - leaves.shape[0])), (0, 0)], mode='constant')

        # Internal nodes: column 3 > 1.
        nodes = enc[i][enc[i, :, 3] > 1]
        nodes = nodes[np.argsort(nodes[:, 1])]

        # Copy the last node's value to balance leaves and nodes.
        # This follows the original notebook behavior.
        nodes = np.append(nodes, nodes[-1].reshape(1, -1), axis=0)
        nodes = np.pad(nodes, [(0, (max_len - nodes.shape[0])), (0, 0)], mode='constant')

        enc_pad.append(np.stack((leaves, nodes), axis=2))

    return np.array(enc_pad)

# Change encoding to order by root age and pad with 0s.
encoding_pad_test = encode_pad_0s_rootage_single(encoding_test)
encoding_pad_calibration = encode_pad_0s_rootage_single(encoding_calibration)

# Verify compatibility with the model input shape: (400, 12, 2).
print("Test encoding:", encoding_pad_test.shape)
print("Calibration encoding:", encoding_pad_calibration.shape)

In [ ]:
# Load the model architecture and trained weights.
serialization_lib.enable_unsafe_deserialization()

MODEL_JSON = (
    "./trained_models/250_350_SIM_SIM_trained_BDEIVH.json"
)

MODEL_WEIGHTS = (
    "./trained_models/250_350_SIM_SIM_trained_BDEIVH.weights.h5"
)

# Necessário para desserializar a Lambda layer do modelo confiável.
serialization_lib.enable_unsafe_deserialization()

with open(MODEL_JSON, "r", encoding="utf-8") as json_file:
    estimator = model_from_json(json_file.read())

estimator.load_weights(MODEL_WEIGHTS)

print("Model loaded successfully.")
print("Model input shape:", estimator.input_shape)


In [ ]:
# Generate predictions for the test and calibration sets.
Y_pred_test = estimator.predict(encoding_pad_test)
Y_pred_calibration = estimator.predict(encoding_pad_calibration)

# Reshape predictions to:
# (n_samples, n_targets, n_quantiles)
Y_pred_test = Y_pred_test.reshape(
    -1,
    len(target_cols),
    len(QUANTILES)
)

Y_pred_calibration = Y_pred_calibration.reshape(
    -1,
    len(target_cols),
    len(QUANTILES)
)

# Column names for lower, central and upper predictions.
pred_cols = []

for col in target_cols:
    pred_cols.extend([
        f"{col}_lower",
        col,
        f"{col}_upper"
    ])

# Convert test predictions to a DataFrame.
predicted_test = pd.DataFrame(
    Y_pred_test.reshape(Y_pred_test.shape[0], -1),
    columns=pred_cols
)

predicted_test.index = Y_test.index

# Convert calibration predictions to a DataFrame.
predicted_calibration = pd.DataFrame(
    Y_pred_calibration.reshape(Y_pred_calibration.shape[0], -1),
    columns=pred_cols
)

predicted_calibration.index = Y_calibration.index

# Verify dimensions.
print("Test predictions:", predicted_test.shape)
print("Calibration predictions:", predicted_calibration.shape)

In [ ]:
# Predict quantiles for calibration and test sets, then apply CQR.

ALPHA = 0.05  # 95% prediction intervals.

print("Calibration prediction shape:", Y_pred_calibration.shape)
print("Test prediction shape:", Y_pred_test.shape)

n_targets = len(target_cols)
n_quantiles = len(QUANTILES)

# Reshape from (n_samples, n_targets * n_quantiles) to (n_samples, n_targets, n_quantiles).
Y_pred_cal_3d = Y_pred_calibration.reshape(-1, n_targets, n_quantiles)
Y_pred_test_3d = Y_pred_test.reshape(-1, n_targets, n_quantiles)

# Quantile positions: lower, median, upper.
lower_idx = 0
median_idx = 1
upper_idx = 2

# CQR conformity score for each calibration example and target:
# score = max(q_lower - y_true, y_true - q_upper)
y_cal_values = Y_calibration[target_cols].values
cal_scores = np.maximum(
    Y_pred_cal_3d[:, :, lower_idx] - y_cal_values,
    y_cal_values - Y_pred_cal_3d[:, :, upper_idx]
)

# Split conformal finite-sample correction.
# This is computed separately for each target parameter.
n_cal = cal_scores.shape[0]
q_level = np.ceil((n_cal + 1) * (1 - ALPHA)) / n_cal
q_level = min(q_level, 1.0)

try:
    q_hat = np.quantile(cal_scores, q_level, axis=0, method='higher')
except TypeError:
    # For older NumPy versions.
    q_hat = np.quantile(cal_scores, q_level, axis=0, interpolation='higher')

print("CQR q_level:", q_level)
print("CQR q_hat per target:")
for col, q in zip(target_cols, q_hat):
    print(f"  {col}: {q}")

# Apply CQR correction to test-set intervals.
Y_pred_test_calibrated = Y_pred_test_3d.copy()
Y_pred_test_calibrated[:, :, lower_idx] = Y_pred_test_3d[:, :, lower_idx] - q_hat
Y_pred_test_calibrated[:, :, upper_idx] = Y_pred_test_3d[:, :, upper_idx] + q_hat

# Convert calibrated predictions to dataframe.
pred_cols = []
for col in target_cols:
    pred_cols.extend([
        f"{col}_lower",
        col,
        f"{col}_upper"
    ])

predicted_test = pd.DataFrame(
    Y_pred_test_calibrated.reshape(Y_pred_test_calibrated.shape[0], -1),
    columns=pred_cols
)
predicted_test.index = Y_test.index

# Optional quick empirical coverage on the held-out test set, in the scaled space.
coverage = {}
for col in target_cols:
    coverage[col] = np.mean(
        (Y_test[col].values >= predicted_test[f"{col}_lower"].values) &
        (Y_test[col].values <= predicted_test[f"{col}_upper"].values)
    )

print("Empirical test coverage after CQR, scaled space:")
for col, cov in coverage.items():
    print(f"  {col}: {cov:.3f}")

predicted_test.head()

In [ ]:
# Restore the original scale of the time-dependent parameters
# and save the true and predicted values to CSV files.
#
# During preprocessing, the parameters d_v and d_h were divided by
# the tree-specific rescaling factor so that they matched the scale
# of the normalized phylogenetic trees used for training.
#
# Before exporting the results, these parameters are multiplied by
# the same rescaling factor to recover their original units.
#
# The remaining parameters were not rescaled during preprocessing
# and therefore require no further transformation.

import os

# Create the output directory for the prediction files.
# No error is raised if the directory already exists.
os.makedirs("./predictions", exist_ok=True)

# Parameters that require inverse rescaling.
cols_to_rescale = [target_4, target_5]

# Tree-specific rescaling factor.
factor = param_test["resc_factor"].to_numpy(dtype=np.float32)

# Create independent copies so that the original DataFrames
# remain unchanged.
Y_test_rescaled = Y_test.copy()
predicted_test_rescaled = predicted_test.copy()

# Restore the original scale of the time-dependent parameters.
for col in cols_to_rescale:

    # Restore the true parameter values.
    Y_test_rescaled[col] = (
        Y_test_rescaled[col].to_numpy() * factor
    )

    # Restore the predicted median.
    predicted_test_rescaled[col] = (
        predicted_test_rescaled[col].to_numpy() * factor
    )

    # Restore the lower prediction limit.
    predicted_test_rescaled[f"{col}_lower"] = (
        predicted_test_rescaled[f"{col}_lower"].to_numpy() * factor
    )

    # Restore the upper prediction limit.
    predicted_test_rescaled[f"{col}_upper"] = (
        predicted_test_rescaled[f"{col}_upper"].to_numpy() * factor
    )

# Save the true parameter values.
Y_test_rescaled.to_csv(
    "./predictions/250_350_SIM_OROV_trained_BDEIVH_true.csv",
    header=True,
    index=True
)

# Save the predicted median and prediction intervals.
predicted_test_rescaled.to_csv(
    "./predictions/250_350_SIM_OROV_trained_BDEIVH_predicted.csv",
    header=True,
    index=True
)

# Confirm that the files were successfully written.
print("Prediction files saved:")
print("./predictions/250_350_SIM_OROV_trained_BDEIVH_true.csv")
print("./predictions/250_350_SIM_OROV_trained_BDEIVH_predicted.csv")